# 02 - Kinematic PCA per phase

**Purpose**: fit PCA separately on each experimental phase's
(subject, contacted) kinematic summary and compare loading patterns
across phases.

**Input**: ``data.AKDdf_agg_contact`` (aggregated kinematics by
contact_group), flattened.

**Caveat**: at N=4 each per-phase PCA has 4 subjects x many features --
PC1 is well-estimated, higher PCs are noisy.


In [ ]:
# parameters
PHASES = ['Baseline', 'Post_Injury_1', 'Post_Injury_2-4', 'Post_Rehab_Test']
N_COMPONENTS = 3
TOP_N_FEATURES = 5  # Features to count as "important" per PC when pooling across phases
FIGSIZE_HEATMAP = (24, 10)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from endpoint_ck_analysis.config import CACHE_DIR, EXAMPLE_OUTPUT_DIR
from endpoint_ck_analysis.data_loader import load_all
from endpoint_ck_analysis.helpers.dimreduce import run_pca_for_phase, align_signs_to_reference
from endpoint_ck_analysis.helpers.plotting import plot_pca_for_phase, stamp_version

EXAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
data = load_all()
agg_flat = data.AKDdf_agg_contact_flat()


## 1. Fit PCA per phase

``run_pca_for_phase`` filters to the phase + contacted reaches, scales,
runs PCA, and returns the fit plus its scores / loadings.


In [ ]:
pcas = {phase: run_pca_for_phase(agg_flat, phase, n_components=N_COMPONENTS) for phase in PHASES}
for phase, (pca, _, eigen, _, _) in pcas.items():
    print(f'{phase}: variance explained = {pca.explained_variance_ratio_.round(3)}')


## 2. Per-phase scree + loading figures

``plot_pca_for_phase`` emits one scree plot and a loading-bars figure
per phase. Saves PNGs for the gallery.


In [ ]:
for phase, (pca, _, eigen, loadings, _) in pcas.items():
    plot_pca_for_phase(pca, eigen, loadings, phase)


## 3. Align PC signs across phases

Per-phase PCA produces PCs with arbitrary sign. Aligning each
non-baseline phase's PCs to Baseline (flipping if Pearson r < 0) so
"positive PC1 loading" means the same thing across phases.


In [ ]:
loadings_baseline = pcas['Baseline'][3]
loadings_by_phase = {'Baseline': loadings_baseline}
for phase in PHASES[1:]:
    loadings_by_phase[phase] = align_signs_to_reference(pcas[phase][3], loadings_baseline)

loadings_concat = pd.concat(loadings_by_phase, axis=0)


## 4. Heatmap: per-PC loadings across phases

Each subplot is one PC. Rows are kinematic features, columns are phases.
Red = positive loading, blue = negative.


In [ ]:
fig, axes = plt.subplots(1, N_COMPONENTS, figsize=FIGSIZE_HEATMAP)
for i, pc in enumerate([f'PC{k+1}' for k in range(N_COMPONENTS)]):
    pc_loadings = loadings_concat.xs(pc, level=1).T
    sns.heatmap(pc_loadings, cmap='RdBu_r', center=0, ax=axes[i],
                cbar_kws={'label': 'Loading'})
    axes[i].set_title(f'{pc} loadings across phases')
plt.tight_layout()
stamp_version(fig, label='02 loadings by phase')
plt.savefig(EXAMPLE_OUTPUT_DIR / '02_loadings_by_phase.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Identify important features (mean |loading| across phases)

Pool each PC's absolute loadings across phases; the top ``TOP_N_FEATURES``
per PC are the kinematic features most consistently structured by the
dominant subject-level variance. Write their union to the cache for
the PLS notebook.


In [ ]:
important_features = {}
for pc in [f'PC{k+1}' for k in range(N_COMPONENTS)]:
    pc_loadings = loadings_concat.xs(pc, level=1).T
    mean_abs = pc_loadings.abs().mean(axis=1)
    top = mean_abs.nlargest(TOP_N_FEATURES)
    important_features[pc] = top
    print(f'\nTop {TOP_N_FEATURES} features on {pc} by mean |loading|:')
    print(top)

all_important = set()
for pc, series in important_features.items():
    all_important.update(series.index)
print(f'\n{len(all_important)} unique important features total')

pd.Series(sorted(all_important), name='feature').to_frame().to_parquet(
    CACHE_DIR / 'important_features.parquet', index=False
)


## 6. Heatmap filtered to important features


In [ ]:
fig, axes = plt.subplots(1, N_COMPONENTS, figsize=FIGSIZE_HEATMAP)
for i, pc in enumerate([f'PC{k+1}' for k in range(N_COMPONENTS)]):
    pc_loadings = loadings_concat.xs(pc, level=1).T
    pc_top = pc_loadings.loc[pc_loadings.index.isin(all_important)]
    sns.heatmap(pc_top, cmap='RdBu_r', center=0, ax=axes[i],
                cbar_kws={'label': 'Loading'})
    axes[i].set_title(f'{pc} loadings (important features)')
plt.tight_layout()
stamp_version(fig, label='02 important features')
plt.savefig(EXAMPLE_OUTPUT_DIR / '02_loadings_important_features.png', dpi=150, bbox_inches='tight')
plt.show()


<!-- INTERPRETATION_BLOCK -->
## How to read this notebook's output

<details>
<summary>What the per-phase kinematic PCA outputs tell you (click to expand)</summary>

**Per-phase scree plots** (one per phase): how variance partitions across
PCs within each phase.

- Similar shapes across the four phases = kinematic structure is
  preserved across injury and recovery; the same axes of variation
  exist before and after.
- Very different scree shapes = the kinematic state space changes with
  phase (e.g., more PCs needed post-injury to capture the same total
  variance, suggesting movement variability increases).

**Per-phase loading bars**: which kinematic features drive each PC at
each phase.

**Sign-aligned cross-phase heatmap** (the headline figure): rows are
features, columns are phases, color = loading on each PC.

- A row where colors are similar across all four phases = that feature
  carries the same role pre-injury, post-injury, and post-rehab.
- A row where colors flip or change magnitude across phases = the
  feature's role in the dominant axis of variation depends on phase.
  These are the features that "mean different things" at different
  experimental timepoints.

**Important features list**: the union across PCs of features with the
highest mean absolute loading. These feed into the PLS Y-block in
notebook 04 -- they're the kinematic features the variance-mapping
flagged as carrying signal.

At N=4 each per-phase PCA has 4 subjects -- PC1 is reasonably stable but
higher PCs are noisy.

</details>
